# Insurance Data Validation Agent
This notebook contains the core logic of the data validation agent, extracted from the Streamlit app.

### 1. The Context
The goal of this notebook is to build an automated **Data Validation and Fraud Detection Pipeline** specifically for insurance datasets. Insurance data is often messy (missing values, inconsistent formatting, outliers). This notebook acts as an "agent" that cleans the raw data, identifies potentially fraudulent claims using both machine learning and business rules, and outputs a clean, structured report.

### 2. Libraries
- **`pandas` (pd):** The powerhouse of Python data analysis for tabular data manipulation.
- **`numpy` (np):** Used for numerical computing and handling missing data (`np.nan`).
- **`os`:** For interacting with the operating system (file paths, directories).
- **`IsolationForest` (scikit-learn):** An unsupervised machine learning algorithm for anomaly detection.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import IsolationForest

### 3. Data Ingestion & Cleaning Methods
**Data Ingestion & EDA:**
- `load_data` & `get_data_summary`: Handles File I/O and converts files into pandas DataFrames.
- `check_missing_values` & `check_duplicates`: Profiles the data to inform the cleaning phase.

**Data Cleaning (`clean_data`):**
- Standardizes various representations of nulls to `np.nan`.
- Drops sparse columns (>80% missing) and columns with zero variance.
- Treats outliers using the **Interquartile Range (IQR) Method** (statistical capping).
- Imputes missing numeric values with the **Median** and text values with the **Mode**.

In [2]:
def load_data(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Please use CSV or Excel.")
    return df

def get_data_summary(df):
    return {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'columns': df.columns.tolist()
    }

def check_missing_values(df):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    return missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

def check_duplicates(df, subset=None):
    duplicates = df[df.duplicated(subset=subset, keep=False)]
    return len(duplicates), duplicates

def clean_data(df):
    df_clean = df.copy()
    
    if 'Product_Type' in df_clean.columns:
        df_clean['Product_Type'] = df_clean['Product_Type'].replace(['unnown', 'Unknown', 'unknown', 'UNKNOWN'], 'Miscellaneous')
        
    missing_strings = ['unknown', 'NA', 'null', 'Null', 'NULL', '?', '??', '???', ' ', '', 'N/A', 'n/a']
    df_clean.replace(missing_strings, np.nan, inplace=True)
    
    for c in ['Written_Premium', 'Claim_Amount', 'Total_Expense']:
        if c in df_clean.columns:
            df_clean[c] = pd.to_numeric(df_clean[c], errors='coerce')
    cols_to_drop = set(['Mystery_Risk_Factor', 'Legacy_System_Code'])
    for col in df_clean.columns:
        if df_clean[col].isnull().mean() > 0.80:
            cols_to_drop.add(col)
            continue
        if df_clean[col].nunique(dropna=True) <= 1:
            cols_to_drop.add(col)
            continue
        is_id_col = str(col).lower().endswith('id') or str(col).lower().endswith('_id')
        if not pd.api.types.is_numeric_dtype(df_clean[col]) and not is_id_col:
            if df_clean[col].count() > 0 and (df_clean[col].nunique(dropna=True) / df_clean[col].count()) > 0.90:
                cols_to_drop.add(col)
    
    df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns], inplace=True)
    
    df_clean.dropna(how='all', inplace=True)
    df_clean.dropna(axis=1, how='all', inplace=True)
    
    for col in df_clean.select_dtypes(include=['float64', 'int64']).columns:
        is_id_col = str(col).lower().endswith('id') or str(col).lower().endswith('_id')
        if not is_id_col and df_clean[col].count() > 0:
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            df_clean[col] = df_clean[col].clip(lower=Q1 - 1.5 * IQR, upper=Q3 + 1.5 * IQR)
            
    for col in df_clean.columns:
        if df_clean[col].isnull().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            else:
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
                
    return df_clean

### 4. Anomaly & Fraud Detection Methods
- `detect_anomalies_isolation_forest`: Uses the **Isolation Forest** model to detect multivariate outliers. It isolates anomalies by building random decision trees, assuming ~5% of data is anomalous.
- `calculate_fraud_score`: Applies **Rule-Based Heuristic Scoring** to flag potentially fraudulent behavior based on hardcoded business logic (e.g., high claim amounts combined with short policy tenure).
- `generate_excel_report`: Uses `openpyxl` to generate a multi-sheet Excel report containing the cleaned data, anomalies, and fraud scores.

In [ ]:
def detect_anomalies_isolation_forest(df, numerical_cols):
    df_subset = df[numerical_cols].fillna(df[numerical_cols].median())
    model = IsolationForest(contamination=0.05, random_state=42)
    predictions = model.fit_predict(df_subset)
    anomaly_mask = predictions == -1
    return df[anomaly_mask]

def calculate_fraud_score(df):
    df_scored = df.copy()
    for col in ['Claim_Frequency', 'Claim_Amount', 'Policy_Tenure_Months', 'Underwriting_Exception_Flag', 'Claim_Exception_Flag']:
        if col in df_scored.columns:
            df_scored[col] = pd.to_numeric(df_scored[col], errors='coerce').fillna(0)
    df_scored['Calculated_Fraud_Score'] = 0
    if 'Claim_Frequency' in df.columns:
        df_scored.loc[df_scored['Claim_Frequency'] > df_scored['Claim_Frequency'].quantile(0.9), 'Calculated_Fraud_Score'] += 20
    if 'Underwriting_Exception_Flag' in df.columns:
        df_scored.loc[df_scored['Underwriting_Exception_Flag'] == 1, 'Calculated_Fraud_Score'] += 25
    if 'Claim_Exception_Flag' in df.columns:
        df_scored.loc[df_scored['Claim_Exception_Flag'] == 1, 'Calculated_Fraud_Score'] += 25
    if 'Claim_Amount' in df.columns and 'Policy_Tenure_Months' in df.columns:
        high_claim = df_scored['Claim_Amount'] > df_scored['Claim_Amount'].quantile(0.8)
        short_tenure = df_scored['Policy_Tenure_Months'] < 12
        df_scored.loc[high_claim & short_tenure, 'Calculated_Fraud_Score'] += 30
    df_scored['Calculated_Fraud_Score'] = df_scored['Calculated_Fraud_Score'].clip(upper=100)
    return df_scored

def generate_excel_report(df_clean, anomalies, fraud_scores, output_dir="reports"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    output_path = os.path.join(output_dir, "validation_report.xlsx")
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_clean.to_excel(writer, sheet_name='Cleaned_Data', index=False)
        anomalies.to_excel(writer, sheet_name='Anomalies', index=False)
        if 'Policy_ID' in fraud_scores.columns:
            fraud_scores[['Policy_ID', 'Calculated_Fraud_Score']].to_excel(writer, sheet_name='Fraud_Scores', index=False)
        else:
            fraud_scores[['Calculated_Fraud_Score']].to_excel(writer, sheet_name='Fraud_Scores', index=False)
    return output_path

### 5. Execution Block
The cell below runs the entire pipeline end-to-end. Replace `"sample_data.csv"` with the actual path to your data.

In [7]:
file_path = r"C:\Users\ASUS\.gemini\antigravity\scratch\CRIP\Dataset\Unclean.xlsx"  # Replace with actual file path

try:
    print("Loading Data...")
    df = load_data(file_path)
    
    print("\n--- Data Summary ---")
    summary = get_data_summary(df)
    print(f"Total Rows: {summary['total_rows']}")
    print(f"Total Columns: {summary['total_columns']}")
    display(df.head())
    
    print("\n--- Checking Missing Values ---")
    missing_df = check_missing_values(df)
    if not missing_df.empty:
        display(missing_df)
    else:
        print("No missing values found!")
        
    print("\n--- Checking Duplicates ---")
    num_dupes, duplicates = check_duplicates(df)
    print(f"Duplicates: {num_dupes} rows")
    
    print("\n--- Cleaning Data ---")
    df_clean = clean_data(df)
    print("Data cleaning complete.")
    
    print("\n--- Detecting Anomalies ---")
    num_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
    num_cols = [col for col in num_cols if not (str(col).lower().endswith('id') or str(col).lower().endswith('_id'))]
    anomalies = detect_anomalies_isolation_forest(df_clean, num_cols)
    print(f"Found {len(anomalies)} anomalies.")
    display(anomalies.head())
    
    print("\n--- Calculating Fraud Scores ---")
    df_scored = calculate_fraud_score(df_clean)
    print("Top 5 Highest Fraud Scores:")
    if 'Policy_ID' in df_scored.columns:
        display(df_scored[['Policy_ID', 'Calculated_Fraud_Score']].sort_values(by='Calculated_Fraud_Score', ascending=False).head())
    else:
        display(df_scored[['Calculated_Fraud_Score']].sort_values(by='Calculated_Fraud_Score', ascending=False).head())
        
    print("\n--- Generating Report ---")
    report_path = generate_excel_report(df_clean, anomalies, df_scored)
    print(f"Report saved to: {report_path}")
    
except FileNotFoundError as e:
    print(e)
    print("Please provide a valid path to your dataset.")

Loading Data...

--- Data Summary ---
Total Rows: 2977
Total Columns: 51


,Policy_ID,Date,Product_Type,Policy_Status,Customer_Segment,Distribution_Channel,Policy_Tenure_Months,Renewal_Count,Written_Premium,Sum_Insured,...,Actual_Loss,State,State_Hazard_Score,Flood_Zone,Cyclone_Zone,Earthquake_Zone,Urban_Rural,Coastal_Flag,Mystery_Risk_Factor,Legacy_System_Code
0,POL10001,2022-01-01 00:00:00,Motor,Active,Individual,Agent,2,0,9630.25,111325.64,...,5431.55,Madhya Pradesh,3,5,1,3,Rural,0,X2,L001
1,POL10068,2022-01-01 00:00:00,Motor,Active,SME,Bancassurance,4,0,16134.56,177438.17,...,13758.87,Goa,6,1,2,5,Rural,1,???,NULL_VAL
2,POL10067,2022-01-01 00:00:00,Motor,Active,Individual,Online,26,1,17295.14,241409.37,...,9869.01,Assam,8,7,3,9,Urban,0,X1,NULL_VAL
3,POL10066,2022-01-01 00:00:00,Motor,Cancelled,SME,Bancassurance,2,0,23691.83,220253.45,...,17891.26,Punjab,2,1,1,3,Rural,0,UNKNOWN,OLD
4,POL10065,2022-01-01 00:00:00,Motor,Active,Individual,Agent,18,1,13418.2,116871.79,...,6865.89,West Bengal,7,9,9,5,Urban,1,???,OLD



--- Checking Missing Values ---


,Missing Count,Missing %
Mystery_Risk_Factor,590,19.818609
Product_Type,89,2.989587
Written_Premium,89,2.989587
Claim_Amount,89,2.989587
Days_Past_Due,89,2.989587
State,89,2.989587



--- Checking Duplicates ---
Duplicates: 0 rows

--- Cleaning Data ---
Data cleaning complete.

--- Detecting Anomalies ---
Found 149 anomalies.


,Policy_ID,Date,Product_Type,Policy_Status,Customer_Segment,Distribution_Channel,Policy_Tenure_Months,Renewal_Count,Sum_Insured,Claim_Count,...,Reinsurer_Rating,Expected_Loss,Actual_Loss,State,State_Hazard_Score,Flood_Zone,Cyclone_Zone,Earthquake_Zone,Urban_Rural,Coastal_Flag
16,POL10053,2022-01-01 00:00:00,Fire,Active,Corporate,Bancassurance,28,1.0,1836650.635,0,...,AAA,45349.99,46779.67,Maharashtra,6,1,4.0,1,Urban,1
39,POL10078,2022-01-01 00:00:00,Property,Active,Corporate,Agent,55,1.0,1689294.960,0,...,A,45349.99,46779.67,Goa,6,3,2.0,1,Urban,1
55,POL10014,2022-01-01 00:00:00,Marine,Active,SME,Agent,30,1.0,1836650.635,2,...,A,45349.99,42323.40,Tamil Nadu,7,6,9.5,6,Rural,1
57,POL10022,2022-01-01 00:00:00,Marine,Active,SME,Online,1,0.0,1836650.635,4,...,A,45349.99,38768.00,Haryana,2,4,1.0,5,Rural,0
185,POL10249,2022-03-01 00:00:00,Marine,Active,Individual,Online,14,1.0,1836650.635,2,...,AAA,43531.70,34465.23,Gujarat,7,5,9.5,1,Urban,1



--- Calculating Fraud Scores ---
Top 5 Highest Fraud Scores:


,Policy_ID,Calculated_Fraud_Score
2964,POL12927,30
2973,POL12918,30
20,POL10069,30
27,POL10090,30
2932,POL12958,30



--- Generating Report ---
Report saved to: reports\validation_report.xlsx
